In [131]:
import sqlite3
import pandas as pd
from pathlib import Path

Within the analysis of this dataset, I aimed to answer several questions including: 
- Sales Trends: How does total revenue trend month-over-month? 
- Delivery Performance: What percentage of orders arrive after the estimated delivery date?
- Customer Location: Which Brazilian states generate the highest volume of sales?
- Payment Choice: Do customers spending more money use more installment payments?
- Review Triggers: How strongly do delivery delays correlate with 1-star review scores?

# Sales Trends

In [132]:
query: str = """
SELECT
op.payment_value,
strftime('%m', oi.order_purchase_timestamp) AS order_month,
strftime('%Y', oi.order_purchase_timestamp) AS order_year
FROM 'order payments' AS op
INNER JOIN 'orders' AS oi ON op.order_id = oi.order_id
GROUP BY order_month, order_year
ORDER BY op.payment_value DESC

            """
conn: sqlite3.Connection = sqlite3.connect("../database/olist.db")
df: pd.DataFrame = pd.read_sql(query, conn)

df

,payment_value,order_month,order_year
0,387.80,01,2017
1,341.09,07,2018
2,283.34,05,2017
3,244.15,01,2018
4,222.03,10,2018
5,197.55,09,2018
6,188.73,11,2017
7,170.57,03,2017
8,157.45,03,2018
9,136.23,09,2016


# Delivery Performance

In [ ]:
query = """
        SELECT
        strftime('%Y', order_purchase_timestamp) AS order_year,
        strftime('%m', order_purchase_timestamp) AS order_month,
        AVG(julianday(order_estimated_delivery_date) - julianday(order_delivered_customer_date)) AS data_diff
        FROM orders
        WHERE order_status = 'delivered'
        GROUP BY order_year, order_month

        """

df: pd.DataFrame = pd.read_sql(query, conn)
df  

,order_year,order_month,data_diff
0,2016,09,-36.324745
1,2016,10,36.076073
2,2016,12,21.336991
3,2017,01,26.861787
4,2017,02,18.680104
5,2017,03,11.781202
6,2017,04,12.431898
7,2017,05,12.962421
8,2017,06,12.010292
9,2017,07,11.724584


# Customer Location

Most profitable city and state pairs

In [134]:
query: str = """
SELECT customer_city, customer_state, COUNT(*) AS CustomerCount
FROM customers 
GROUP BY customer_city, customer_state
ORDER BY COUNT(*) DESC
"""
conn: sqlite3.Connection = sqlite3.connect("../database/olist.db")
df: pd.DataFrame = pd.read_sql(query, conn)

df

,customer_city,customer_state,CustomerCount
0,sao paulo,SP,15540
1,rio de janeiro,RJ,6882
2,belo horizonte,MG,2773
3,brasilia,DF,2131
4,curitiba,PR,1521
...,...,...,...
4305,vitoria do jari,AP,1
4306,vitorino,PR,1
4307,vitorinos,MG,1
4308,wagner,BA,1


Most profitable states

In [135]:
query: str = """
SELECT customer_state, COUNT(*) AS CustomerCount
FROM customers 
GROUP BY customer_state
ORDER BY COUNT(*) DESC
"""
conn: sqlite3.Connection = sqlite3.connect("../database/olist.db")
df: pd.DataFrame = pd.read_sql(query, conn)

df

,customer_state,CustomerCount
0,SP,41746
1,RJ,12852
2,MG,11635
3,RS,5466
4,PR,5045
5,SC,3637
6,BA,3380
7,DF,2140
8,ES,2033
9,GO,2020


# Bonus: Most purchases dates

In [141]:
query: str = """
SELECT DATE(order_purchase_timestamp) AS purchase_date,
       COUNT(*) AS order_count
FROM orders
GROUP BY DATE(order_purchase_timestamp)
ORDER BY COUNT(*) DESC;

"""
conn: sqlite3.Connection = sqlite3.connect("../database/olist.db")
df: pd.DataFrame = pd.read_sql(query, conn)

df

,purchase_date,order_count
0,2017-11-24,1176
1,2017-11-25,499
2,2017-11-27,403
3,2017-11-26,391
4,2017-11-28,380
...,...,...
629,2016-10-02,1
630,2016-09-15,1
631,2016-09-13,1
632,2016-09-05,1


99441 orders

# Bonus: Most common delivery dates

In [139]:
query = """
        SELECT DATE(order_delivered_customer_date),
        COUNT(*)
        FROM orders
        GROUP BY DATE(order_delivered_customer_date)
        ORDER BY COUNT(*) DESC
        """

df: pd.DataFrame = pd.read_sql(query, conn)
df

,DATE(order_delivered_customer_date),COUNT(*)
0,NaN,2965
1,2018-08-27,446
2,2018-08-13,442
3,2018-05-14,434
4,2018-05-21,431
...,...,...
641,2016-11-21,1
642,2016-11-14,1
643,2016-11-05,1
644,2016-10-30,1


# Bonus: Most popular categories

In [138]:
query = """
        SELECT t.product_category_name_english,
        COUNT(*) AS count_orders
        FROM products AS p
        INNER JOIN 'order items' AS oi ON p.product_id = oi.product_id
        INNER JOIN 'product category name translation' AS t ON p.product_category_name = t.product_category_name
        GROUP BY t.product_category_name_english
        ORDER BY COUNT(*) DESC
        """

df: pd.DataFrame = pd.read_sql(query, conn)
df

,product_category_name_english,count_orders
0,bed_bath_table,11115
1,health_beauty,9670
2,sports_leisure,8641
3,furniture_decor,8334
4,computers_accessories,7827
...,...,...
66,arts_and_craftmanship,24
67,la_cuisine,14
68,cds_dvds_musicals,14
69,fashion_childrens_clothes,8


In [ ]:
def read_SQL_Query(filePath) -> pd.DataFrame:
    with open(Path(filePath), "r") as file:
        query: str = file.read()

    df: pd.DataFrame = pd.read_sql(query, conn)
    return df

customer_count: pd.DataFrame = read_SQL_Query(r"..\sql\test.sql")
customer_count